# Proyecto: Shopper Intention

## 1) Preparar los datos
Cargar dataset, separar `Revenue`, transformar variables categóricas y dividir en train/test (80/20).

Cargamos dependencias.

In [16]:
import pandas as pd
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

Declaramos variables globales.

In [17]:
DATA_URL = "https://raw.githubusercontent.com/sharmaroshan/Online-Shoppers-Purchasing-Intention/master/online_shoppers_intention.csv"
TARGET = "Revenue"
TEST_SIZE = 0.20
RANDOM_STATE = 42
CATEGORICAL_COLS = ["Month", "VisitorType"]
BOOL_MAP = {"TRUE": 1, "FALSE": 0, True: 1, False: 0}

Cargamos el dataset.

In [18]:
df = pd.read_csv(DATA_URL)
print(f"Filas: {df.shape[0]:,} | Columnas: {df.shape[1]}")
df.head()

Filas: 12,330 | Columnas: 18


,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,False,False
1,0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,Feb,2,2,1,2,Returning_Visitor,False,False
2,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,False,False
3,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,Feb,3,2,2,4,Returning_Visitor,False,False
4,0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,Feb,3,3,1,4,Returning_Visitor,True,False


Limpiamos y transformamos los datos.

In [19]:
# Limpieza y transformación mínima
if df.isnull().sum().sum() > 0:
    raise ValueError("El dataset contiene valores nulos.")

y = df[TARGET].map(BOOL_MAP)
if y.isnull().any():
    raise ValueError("Revenue contiene valores no esperados.")

X = df.drop(columns=[TARGET]).copy()
X["Weekend"] = X["Weekend"].map(BOOL_MAP)
if X["Weekend"].isnull().any():
    raise ValueError("Weekend contiene valores no esperados.")

X = pd.get_dummies(X, columns=CATEGORICAL_COLS, drop_first=True)

Separamos en entrenamiento y prueba (80/20).

In [20]:
# train: para entrenar el modelo | test: para evaluarlo con datos no vistos
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

Validamos tamaños y proporción de compra.

In [12]:
print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")
print(f"y_train: {y_train.shape} | y_test: {y_test.shape}")
print(f"Tasa de compra train: {y_train.mean():.3f}")
print(f"Tasa de compra test:  {y_test.mean():.3f}")

X_train: (9864, 26) | X_test: (2466, 26)
y_train: (9864,) | y_test: (2466,)
Tasa de compra train: 0.155
Tasa de compra test:  0.155


## 2) Probar Árbol de Decisión
Entrenamos un modelo simple y revisamos métricas clave de negocio.

Entrenamos y evaluamos el árbol.

In [21]:
tree_model = DecisionTreeClassifier(max_depth=4, random_state=RANDOM_STATE)
tree_model.fit(X_train, y_train)

y_pred_tree = tree_model.predict(X_test)

acc_tree = accuracy_score(y_test, y_pred_tree)
precision_tree = precision_score(y_test, y_pred_tree, zero_division=0)
recall_tree = recall_score(y_test, y_pred_tree, zero_division=0)
cm_tree = confusion_matrix(y_test, y_pred_tree)

print(f"Accuracy:  {acc_tree:.4f}")
print(f"Precision: {precision_tree:.4f}")
print(f"Recall:    {recall_tree:.4f}")
print("\nMatriz de confusión [ [TN, FP], [FN, TP] ]:")
print(cm_tree)

Accuracy:  0.8962
Precision: 0.7739
Recall:    0.4660

Matriz de confusión [ [TN, FP], [FN, TP] ]:
[[2032   52]
 [ 204  178]]


Analizamos falsos positivos y falsos negativos.

In [22]:
tn, fp, fn, tp = cm_tree.ravel()

print(f"Falsos positivos (FP): {fp}")
print("Impacto: descuentos a clientes que no iban a comprar.")

print(f"Falsos negativos (FN): {fn}")
print("Impacto: ventas perdidas por no incentivar a clientes con intención de compra.")

Falsos positivos (FP): 52
Impacto: descuentos a clientes que no iban a comprar.
Falsos negativos (FN): 204
Impacto: ventas perdidas por no incentivar a clientes con intención de compra.


Revisamos la variable más importante del árbol.

In [23]:
feature_importance = pd.Series(tree_model.feature_importances_, index=X_train.columns).sort_values(ascending=False)

top_feature = feature_importance.index[0]
top_score = feature_importance.iloc[0]

print(f"Variable más importante: {top_feature} ({top_score:.4f})")
feature_importance.head(10)

Variable más importante: PageValues (0.8149)


,0
PageValues,0.814930
BounceRates,0.075522
Month_Nov,0.038881
ProductRelated_Duration,0.028895
Administrative,0.024286
ExitRates,0.010611
ProductRelated,0.003648
Month_Sep,0.003225
Administrative_Duration,0.000000
Informational_Duration,0.000000


## 3) Probar Random Forest
Pendiente.
## 4) Conclusión
Pendiente.